# LeetCode 3317 — Find the Number of Possible Ways for an Event bằng Python

- Link: https://leetcode.com/problems/find-the-number-of-possible-ways-for-an-event/
- Độ khó: **Hard**
- Chủ đề: **Tổ hợp đếm**: chia `n` người vào đúng `j` sân khấu khác rỗng — **số Stirling loại hai** `S(n, j)`, chọn sân khấu có thứ tự `x!/(x − j)!`, chấm điểm `y^j`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numberOfWays(self, n: int, x: int, y: int) -> int:
```

## 1. Mục tiêu học

1. Tách một "sự kiện" thành hai phần độc lập: (a) phân `n` nghệ sĩ vào `x` sân khấu; (b) chấm điểm `[1, y]` cho **từng ban nhạc** (= sân khấu **khác rỗng**).
2. Gom theo `j` = số sân khấu khác rỗng: số cách (a) có đúng `j` sân khấu khác rỗng là `S(n, j) · x · (x − 1) ⋯ (x − j + 1)`; phần (b) có `y^j` cách.
3. Tính số Stirling loại hai bằng DP `S(i, j) = j · S(i−1, j) + S(i−1, j−1)` trong `O(n²)`.
4. Cách thay thế: DP trực tiếp theo nghệ sĩ (sân khấu mới / sân khấu cũ), hoặc bao hàm–loại trừ cho toàn ánh.
5. Kiểm chứng bằng vét cạn `x^n` cách phân.

## 2. Đề bài & ví dụ

Có `n` nghệ sĩ; mỗi người được phân vào một trong `x` sân khấu (có thể có sân khấu trống). Những người cùng sân khấu tạo thành một ban nhạc. Sau đó mỗi **ban nhạc** được chấm một điểm trong `[1, y]`. Hai sự kiện khác nhau nếu có nghệ sĩ nào được phân khác sân khấu, hoặc có ban nhạc nào nhận điểm khác. Đếm số sự kiện, modulo `10^9 + 7`.

**Ví dụ 1:** `n = 1, x = 2, y = 3` → `6`. **Ví dụ 2:** `n = 5, x = 2, y = 1` → `32`. **Ví dụ 3:** `n = 3, x = 3, y = 4` → `684`.

**Ràng buộc:** `1 ≤ n, x, y ≤ 1000`.

In [1]:
from typing import List
from itertools import product
from math import comb, perm
MOD = 10 ** 9 + 7

examples = [(1, 2, 3, 6), (5, 2, 1, 32), (3, 3, 4, 684)]
for n, x, y, exp in examples:
    print(f"n = {n}, x = {x}, y = {y} -> kỳ vọng {exp}")

n = 1, x = 2, y = 3 -> kỳ vọng 6
n = 5, x = 2, y = 1 -> kỳ vọng 32
n = 3, x = 3, y = 4 -> kỳ vọng 684


## 3. Kiến thức nền

### 3.1. Số Stirling loại hai

`S(n, j)` = số cách chia `n` phần tử **có nhãn** thành `j` nhóm **không nhãn**, khác rỗng. Truy hồi theo phần tử cuối: hoặc nó tạo nhóm mới (`S(n−1, j−1)`), hoặc nhập vào một trong `j` nhóm đã có (`j · S(n−1, j)`).

In [2]:
def stirling2_table(N, K, mod=None):
    S = [[0] * (K + 1) for _ in range(N + 1)]
    S[0][0] = 1
    for i in range(1, N + 1):
        for j in range(1, min(i, K) + 1):
            v = j * S[i - 1][j] + S[i - 1][j - 1]
            S[i][j] = v % mod if mod else v
    return S

S = stirling2_table(5, 5)
for i in range(1, 6):
    print(f"S({i}, ·) =", S[i][1:i + 1])

S(1, ·) = [1]
S(2, ·) = [1, 1]
S(3, ·) = [1, 3, 1]
S(4, ·) = [1, 7, 6, 1]
S(5, ·) = [1, 15, 25, 10, 1]


### 3.2. Gán nhóm cho sân khấu

Có `j` nhóm (chưa nhãn) và `x` sân khấu: gán mỗi nhóm một sân khấu **khác nhau** ⇒ `x · (x − 1) ⋯ (x − j + 1) = P(x, j)` (bằng 0 nếu `j > x`). Vậy số cách phân có **đúng** `j` sân khấu khác rỗng là `S(n, j) · P(x, j)`.

Kiểm tra: `Σ_j S(n, j) · P(x, j) = x^n` (mọi cách phân).

In [3]:
for n, x in [(3, 2), (4, 3), (5, 5), (6, 2)]:
    S = stirling2_table(n, n)
    print(n, x, sum(S[n][j] * perm(x, j) for j in range(n + 1)), '==', x ** n)

3 2 8 == 8
4 3 81 == 81
5 5 3125 == 3125
6 2 64 == 64


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 3: `n = 3, x = 3, y = 4`.

| `j` (số ban) | `S(3, j)` | `P(3, j)` | `y^j` | tích |
|---|---|---|---|---|
| 1 | 1 | 3 | 4 | 12 |
| 2 | 3 | 6 | 16 | 288 |
| 3 | 1 | 6 | 64 | 384 |

Tổng `12 + 288 + 384 = 684` ✓. Điểm mấu chốt: **chỉ ban nhạc thật** (sân khấu có người) mới được chấm điểm.

## 5. Cách 1 — Brute Force: mọi cách phân sân khấu

In [4]:
def brute(n, x, y):
    total = 0
    for assign in product(range(x), repeat=n):
        total += y ** len(set(assign))
    return total % MOD

for n, x, y, exp in examples:
    print(brute(n, x, y), '| kỳ vọng', exp)

6 | kỳ vọng 6
32 | kỳ vọng 32
684 | kỳ vọng 684


**Độ phức tạp:** `O(x^n · n)`.

## 6. Ý tưởng thuật toán / bước đột phá

```
ans = Σ_{j=1}^{min(n, x)} S(n, j) · P(x, j) · y^j   (mod p)
```

- `S(n, j)` qua bảng DP `O(n · min(n, x))` — có thể chỉ giữ một hàng.
- `P(x, j)` tích lũy: `P(x, j) = P(x, j−1) · (x − j + 1)`.
- `y^j` tích lũy.

Tổng `O(n · min(n, x)) ≤ 10^6`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP theo nghệ sĩ: sân khấu mới hay cũ

`dp[i][j]` = số cách phân `i` nghệ sĩ đầu sao cho đã dùng đúng `j` sân khấu (có nhãn). Nghệ sĩ thứ `i + 1`: vào một trong `j` sân khấu đã dùng, hoặc một trong `x − j` sân khấu mới.

In [5]:
def dp_performers(n, x, y):
    K = min(n, x)
    dp = [0] * (K + 1)
    dp[0] = 1
    for _ in range(n):
        nd = [0] * (K + 1)
        for j in range(K + 1):
            if dp[j]:
                nd[j] = (nd[j] + dp[j] * j) % MOD
                if j < K:
                    nd[j + 1] = (nd[j + 1] + dp[j] * (x - j)) % MOD
        dp = nd
    return sum(dp[j] * pow(y, j, MOD) for j in range(1, K + 1)) % MOD

for n, x, y, exp in examples:
    print(dp_performers(n, x, y), '| kỳ vọng', exp)

6 | kỳ vọng 6
32 | kỳ vọng 32
684 | kỳ vọng 684


`dp[n][j]` chính bằng `S(n, j) · P(x, j)` — cách 2 gộp sẵn hai thừa số.

### 7.2. Cách 3 — Stirling + chỉnh hợp (lời giải chính)

In [6]:
def stirling_solution(n, x, y):
    K = min(n, x)
    row = [1] + [0] * K                     # S(0, ·)
    for i in range(1, n + 1):
        for j in range(min(i, K), 0, -1):  # cập nhật tại chỗ từ phải sang trái
            row[j] = (j * row[j] + row[j - 1]) % MOD
        row[0] = 0
    ans, P, Y = 0, 1, 1
    for j in range(1, K + 1):
        P = P * (x - j + 1) % MOD
        Y = Y * y % MOD
        ans = (ans + row[j] * P % MOD * Y) % MOD
    return ans

for n, x, y, exp in examples:
    print(stirling_solution(n, x, y), '| kỳ vọng', exp)

6 | kỳ vọng 6
32 | kỳ vọng 32
684 | kỳ vọng 684


Cập nhật một hàng **từ phải sang trái** để `row[j − 1]` vẫn là giá trị của hàng trước.

### 7.3. Cách 4 — Bao hàm–loại trừ cho toàn ánh

Số cách phân `n` nghệ sĩ lên **đúng** một tập `j` sân khấu cho trước (mọi sân khấu đều có người) là `surj(n, j) = Σ_{i=0}^{j} (−1)^i C(j, i) (j − i)^n = j! · S(n, j)`. Vậy `ans = Σ_j C(x, j) · surj(n, j) · y^j`.

In [7]:
def inclusion_exclusion(n, x, y):
    K = min(n, x)
    fact = [1] * (K + 1)
    for i in range(1, K + 1):
        fact[i] = fact[i - 1] * i % MOD
    inv = [pow(f, MOD - 2, MOD) for f in fact]
    C = lambda a, b: fact[a] * inv[b] % MOD * inv[a - b] % MOD
    ans = 0
    Cx = 1                                    # C(x, j) tích lũy
    pw = [pow(t, n, MOD) for t in range(K + 1)]
    for j in range(1, K + 1):
        Cx = Cx * (x - j + 1) % MOD * pow(j, MOD - 2, MOD) % MOD
        surj = 0
        for i in range(j + 1):
            term = C(j, i) * pw[j - i] % MOD
            surj = (surj - term) % MOD if i & 1 else (surj + term) % MOD
        ans = (ans + Cx * surj % MOD * pow(y, j, MOD)) % MOD
    return ans

for n, x, y, exp in examples:
    print(inclusion_exclusion(n, x, y), '| kỳ vọng', exp)

6 | kỳ vọng 6
32 | kỳ vọng 32
684 | kỳ vọng 684


Độ phức tạp `O(K²)` — cùng bậc với cách 3, nhưng minh họa mối liên hệ `surj(n, j) = j! S(n, j)`.

## 8. TRACE từng bước

Trace cách 3 với `n = 4, x = 3, y = 2`: in hàng Stirling và từng số hạng.

In [8]:
def trace(n, x, y):
    S = stirling2_table(n, n)
    print(f"S({n}, j) =", S[n][1:])
    total = 0
    for j in range(1, min(n, x) + 1):
        t = S[n][j] * perm(x, j) * y ** j
        total += t
        print(f"  j = {j}: S = {S[n][j]:3d}, P({x},{j}) = {perm(x, j):3d}, y^j = {y ** j:3d} -> {t}")
    print('tổng:', total, '| vét cạn:', brute(n, x, y))

trace(4, 3, 2)

S(4, j) = [1, 7, 6, 1]
  j = 1: S =   1, P(3,1) =   3, y^j =   2 -> 6
  j = 2: S =   7, P(3,2) =   6, y^j =   4 -> 168
  j = 3: S =   6, P(3,3) =   6, y^j =   8 -> 288
tổng: 462 | vét cạn: 462


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def numberOfWays(self, n: int, x: int, y: int) -> int:
        MOD = 10 ** 9 + 7
        K = min(n, x)
        row = [0] * (K + 1)
        row[0] = 1
        for i in range(1, n + 1):
            for j in range(min(i, K), 0, -1):
                row[j] = (j * row[j] + row[j - 1]) % MOD
            row[0] = 0
        ans = 0
        P = 1
        Y = 1
        for j in range(1, K + 1):
            P = P * (x - j + 1) % MOD
            Y = Y * y % MOD
            ans = (ans + row[j] * P % MOD * Y) % MOD
        return ans

for n, x, y, exp in examples:
    got = Solution().numberOfWays(n, x, y)
    print(got, 'OK' if got == exp else 'SAI')

6 OK
32 OK
684 OK


## 10. Vì sao thuật toán đúng

1. **Phân hoạch theo `j`:** mỗi sự kiện có đúng một giá trị `j` = số sân khấu khác rỗng (`1 ≤ j ≤ min(n, x)`).
2. **Đếm cách phân có đúng `j` sân khấu khác rỗng:** một cách phân ⇔ (một phân hoạch `n` nghệ sĩ thành `j` nhóm) + (một đơn ánh từ `j` nhóm vào `x` sân khấu). Phân hoạch có `S(n, j)` cách, đơn ánh `P(x, j)` cách; song ánh vì từ cách phân ta khôi phục duy nhất các nhóm và sân khấu của chúng.
3. **Chấm điểm:** `j` ban nhạc, mỗi ban `y` lựa chọn, độc lập ⇒ `y^j`.
4. **Truy hồi Stirling:** xét phần tử cuối (mục 3.1).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(x^n · n)` | `O(n)` |
| 2. DP theo nghệ sĩ | `O(n · min(n, x))` | `O(min(n, x))` |
| 3. Stirling + chỉnh hợp | `O(n · min(n, x))` | `O(min(n, x))` |
| 4. Bao hàm–loại trừ | `O(min(n, x)² + K log p)` | `O(K)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'dp_performers': dp_performers, 'stirling': stirling_solution, 'incl_excl': inclusion_exclusion,
         'submit': lambda n, x, y: Solution().numberOfWays(n, x, y)}
edge = [(1, 1, 1, 1), (4, 1, 7, 7), (2, 2, 2, 12), (3, 5, 1, 125), (2, 3, 5, 3 * 5 + 6 * 25)]
for n, x, y, exp in examples + edge:
    assert brute(n, x, y) == exp, (n, x, y, brute(n, x, y))
    for name, f in impls.items():
        assert f(n, x, y) == exp, (name, n, x, y)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Ca `y = 1`: điểm không tạo khác biệt ⇒ đáp án `x^n` (ví dụ 2: `2^5 = 32`; ca `(3, 5, 1)`: `125`). Ca `x = 1`: mọi người một ban ⇒ `y`.

### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3317)
cnt = 0
for _ in range(300):
    n, x, y = rng.randint(1, 6), rng.randint(1, 5), rng.randint(1, 6)
    exp = brute(n, x, y)
    for name, f in impls.items():
        assert f(n, x, y) == exp, (name, n, x, y)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ (n, x, y).')

Khớp vét cạn trên 300 bộ (n, x, y).


### 12.3. Ca lớn `n = x = y = 1000`

In [12]:
import time
for n, x, y in [(1000, 1000, 1000), (1000, 7, 1000), (7, 1000, 1000), (1000, 1000, 1)]:
    t = time.time(); a = Solution().numberOfWays(n, x, y); t1 = time.time() - t
    t = time.time(); b = dp_performers(n, x, y); t2 = time.time() - t
    assert a == b
    print(f"n = {n:4d}, x = {x:4d}, y = {y:4d}: {a:10d} | nộp {t1:.2f}s | DP nghệ sĩ {t2:.2f}s")
assert Solution().numberOfWays(1000, 1000, 1) == pow(1000, 1000, MOD)
print('y = 1: khớp x^n.')

n = 1000, x = 1000, y = 1000:  295964505 | nộp 0.06s | DP nghệ sĩ 0.13s
n = 1000, x =    7, y = 1000:  712692310 | nộp 0.00s | DP nghệ sĩ 0.00s
n =    7, x = 1000, y = 1000:  210775311 | nộp 0.00s | DP nghệ sĩ 0.00s


n = 1000, x = 1000, y =    1:  524700271 | nộp 0.05s | DP nghệ sĩ 0.13s


y = 1: khớp x^n.


Kiểm tra độc lập thêm: với `y` tùy ý, `Σ_j S(n, j) P(x, j) y^j` là giá trị tại `y` của một đa thức; khi `x = 1` đáp án là `y`, khi `n = 1` đáp án là `x · y`.

In [13]:
for y in [1, 2, 999]:
    assert Solution().numberOfWays(1000, 1, y) == y
    assert Solution().numberOfWays(1, 1000, y) == 1000 * y % MOD
print('x = 1 -> y; n = 1 -> x·y: đúng.')

x = 1 -> y; n = 1 -> x·y: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chấm điểm cả sân khấu trống

In [14]:
def wrong_score_empty(n, x, y):
    return pow(x, n, MOD) * pow(y, x, MOD) % MOD        # SAI: y^x thay vì y^(số ban)

print('sai :', wrong_score_empty(1, 2, 3), '| đúng:', Solution().numberOfWays(1, 2, 3))

sai : 18 | đúng: 6


**Vì sao sai:** sân khấu trống không có ban nhạc, không được chấm. Ví dụ 1: chỉ 1 ban ⇒ `2 · 3 = 6`, không phải `2 · 9`.

### Lỗi 2 — Quên gán nhóm vào sân khấu (chỉ dùng `S(n, j)`)

In [15]:
def wrong_no_stage_choice(n, x, y):
    S = stirling2_table(n, n, MOD)
    return sum(S[n][j] * pow(y, j, MOD) for j in range(1, min(n, x) + 1)) % MOD     # SAI

print('sai :', wrong_no_stage_choice(1, 2, 3), '| đúng:', Solution().numberOfWays(1, 2, 3))

sai : 3 | đúng: 6


**Vì sao sai:** `S(n, j)` coi các nhóm **không nhãn**, nhưng sân khấu có nhãn — người duy nhất vào sân khấu 1 hay 2 là hai sự kiện khác nhau.

### Lỗi 3 — Dùng tổ hợp `C(x, j)` thay vì chỉnh hợp `P(x, j)`

In [16]:
def wrong_comb(n, x, y):
    S = stirling2_table(n, n, MOD)
    return sum(S[n][j] * comb(x, j) * pow(y, j, MOD) for j in range(1, min(n, x) + 1)) % MOD   # SAI

print('sai :', wrong_comb(3, 3, 4), '| đúng:', Solution().numberOfWays(3, 3, 4))

sai : 220 | đúng: 684


**Vì sao sai:** chọn **tập** `j` sân khấu (`C(x, j)`) rồi ghép với các nhóm không nhãn còn thiếu `j!` cách ghép nhóm–sân khấu. `C(x, j) · j! = P(x, j)`. (Với `j = 1` hai cách trùng nhau nên ví dụ 1 không lộ lỗi.)

### Lỗi 4 — Tính `P(x, j) = x!/(x − j)!` bằng bảng giai thừa khi `j > x` (chỉ số âm)

In [17]:
def wrong_negative_index(n, x, y):
    N = max(n, x)
    fact = [1] * (N + 1)
    for i in range(1, N + 1):
        fact[i] = fact[i - 1] * i % MOD
    S = stirling2_table(n, n, MOD)
    total = 0
    for j in range(1, n + 1):                                   # SAI: không chặn j <= x
        P = fact[x] * pow(fact[x - j], MOD - 2, MOD) % MOD      # x - j < 0 -> fact[-1] (phần tử cuối!)
        total += S[n][j] * P * pow(y, j, MOD)
    return total % MOD

print('sai :', wrong_negative_index(3, 2, 1), '| đúng:', Solution().numberOfWays(3, 2, 1))

sai : 333333344 | đúng: 8


**Vì sao sai:** khi `j > x` phải có `P(x, j) = 0`, nhưng Python chấp nhận chỉ số âm `fact[−1]` mà không báo lỗi. Với `n = 3, x = 2`, số hạng `j = 3` không tồn tại nhưng bị cộng vào. Luôn chặn `j ≤ min(n, x)` hoặc tính `P` bằng tích lũy `(x − j + 1)` (tự bằng 0 khi `j = x + 1`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Phân phần tử có nhãn vào hộp có nhãn, đếm theo số hộp khác rỗng" ⇒ `S(n, j) · P(x, j)`.
- Một đại lượng chỉ phụ thuộc **số hộp khác rỗng** (ở đây `y^j`) ⇒ gom theo `j`.
- Kiểm tra nhanh bằng đẳng thức `Σ_j S(n, j) P(x, j) = x^n` (đặt `y = 1`).

## 15. Biến thể của bài

1. Mọi sân khấu **phải** có người ⇒ chỉ số hạng `j = x`: `S(n, x) · x! · y^x`.
2. Sân khấu **không nhãn** ⇒ `Σ_j S(n, j) · y^j` (điểm vẫn gán cho từng ban).
3. Mỗi ban có ít nhất `m` người ⇒ Stirling "kết hợp" (associated) `S_m(n, j)`, DP `S(i, j) = j S(i−1, j) + C(i−1, m−1) S(i−m, j−1)`.
4. Điểm của các ban phải **khác nhau** ⇒ thay `y^j` bằng `P(y, j)`.

In [18]:
# Biến thể 1, 2, 4
def all_stages_used(n, x, y):
    if x > n:
        return 0
    S = stirling2_table(n, x, MOD)
    return S[n][x] * perm(x, x) % MOD * pow(y, x, MOD) % MOD

def unlabeled_stages(n, y):
    S = stirling2_table(n, n, MOD)
    return sum(S[n][j] * pow(y, j, MOD) for j in range(1, n + 1)) % MOD

def distinct_scores(n, x, y):
    S = stirling2_table(n, n, MOD)
    return sum(S[n][j] * perm(x, j) * perm(y, j) for j in range(1, min(n, x) + 1)) % MOD

rng = random.Random(15)
for _ in range(150):
    n, x, y = rng.randint(1, 5), rng.randint(1, 4), rng.randint(1, 5)
    e1 = e4 = 0
    for assign in product(range(x), repeat=n):
        j = len(set(assign))
        e1 += (j == x) * y ** j
        e4 += perm(y, j)
    assert all_stages_used(n, x, y) == e1 % MOD
    assert distinct_scores(n, x, y) == e4 % MOD
assert unlabeled_stages(3, 1) == 5          # số Bell B(3) = 5 khi y = 1
print('Biến thể 1, 4 khớp vét cạn; biến thể 2 với y = 1 cho số Bell.')

Biến thể 1, 4 khớp vét cạn; biến thể 2 với y = 1 cho số Bell.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi cách phân | `O(x^n n)` | Oracle |
| 2 | DP nghệ sĩ (sân khấu mới/cũ) | `O(n K)` | Không cần biết Stirling |
| 3 | Stirling + chỉnh hợp | `O(n K)` | **Nộp bài** |
| 4 | Bao hàm–loại trừ | `O(K²)` | Liên hệ toàn ánh |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `n = 2, x = 1, y = 5`.

**Bài 2 (tính tay):** `n = 2, x = 2, y = 3`.

**Bài 3 (điền TODO):** đếm số sự kiện mà **có ít nhất một ban được điểm tối đa `y`**.

**Bài 4 (tự viết lại):** biến thể 3 — mỗi ban có ít nhất `m` người.

**Bài 5 (mini challenge):** trả về giá trị kỳ vọng của số ban nhạc nếu mỗi người chọn sân khấu ngẫu nhiên đều (dạng phân số tối giản).

In [19]:
def at_least_one_max_todo(n, x, y):
    # TODO
    raise NotImplementedError

def min_band_size_todo(n, x, y, m):
    # TODO
    raise NotImplementedError

def expected_bands_todo(n, x):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một sân khấu duy nhất. (2) Một ban. (3) `5`.

**Bài 2.** (1) `j = 1`: `S = 1`, `P = 2`, `y = 3` ⇒ 6. (2) `j = 2`: `1 · 2 · 9 = 18`. (3) `24`.

**Bài 3.** (1) Phần bù: không ban nào được `y` ⇒ `(y − 1)^j`. (2) `Σ_j S P (y^j − (y−1)^j)`. (3) Vét cạn.

**Bài 4.** (1) DP Stirling kết hợp. (2) Hoặc DP nghệ sĩ không áp được trực tiếp — dùng công thức với `C(i−1, m−1)`. (3) Vét cạn với `n ≤ 6`.

**Bài 5.** (1) Tuyến tính của kỳ vọng. (2) Sân khấu `s` khác rỗng với xác suất `1 − (1 − 1/x)^n`. (3) `x · (1 − ((x−1)/x)^n)`.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().numberOfWays(2, 1, 5) == 5
assert Solution().numberOfWays(2, 2, 3) == 24 == brute(2, 2, 3)

def at_least_one_max(n, x, y):
    S = stirling2_table(n, n, MOD)
    return sum(S[n][j] * perm(x, j) * (pow(y, j, MOD) - pow(y - 1, j, MOD)) for j in range(1, min(n, x) + 1)) % MOD

rng = random.Random(3)
for _ in range(150):
    n, x, y = rng.randint(1, 5), rng.randint(1, 4), rng.randint(1, 4)
    exp = sum(y ** len(set(a)) - (y - 1) ** len(set(a)) for a in product(range(x), repeat=n))
    assert at_least_one_max(n, x, y) == exp % MOD
print('Bài 1: 5 | Bài 2: 24 | Bài 3 khớp vét cạn.')

Bài 1: 5 | Bài 2: 24 | Bài 3 khớp vét cạn.


In [21]:
# Bài 4, 5
from fractions import Fraction
def min_band_size(n, x, y, m):
    S = [[0] * (n + 1) for _ in range(n + 1)]
    S[0][0] = 1
    for i in range(1, n + 1):
        for j in range(1, n + 1):
            v = j * S[i - 1][j]
            if i >= m:
                v += comb(i - 1, m - 1) * S[i - m][j - 1]
            S[i][j] = v
    return sum(S[n][j] * perm(x, j) * y ** j for j in range(1, min(n, x) + 1)) % MOD

def expected_bands(n, x):
    return x * (1 - Fraction(x - 1, x) ** n)

rng = random.Random(4)
for _ in range(150):
    n, x, y, m = rng.randint(1, 6), rng.randint(1, 4), rng.randint(1, 3), rng.randint(1, 3)
    exp = 0
    for a in product(range(x), repeat=n):
        sizes = [a.count(s) for s in set(a)]
        if min(sizes) >= m:
            exp += y ** len(sizes)
    assert min_band_size(n, x, y, m) == exp % MOD
    assert expected_bands(n, x) == Fraction(sum(len(set(a)) for a in product(range(x), repeat=n)), x ** n)
print('Bài 4, 5 khớp vét cạn. Ví dụ E[số ban] với n = 3, x = 3:', expected_bands(3, 3))

Bài 4, 5 khớp vét cạn. Ví dụ E[số ban] với n = 3, x = 3: 19/9


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
row = Stirling S(n, ·) (cập nhật tại chỗ từ phải sang trái)
ans = Σ_{j=1}^{min(n,x)} row[j] · P(x, j) · y^j       # P(x,j) = P(x,j-1)·(x-j+1)
```

**Checklist:**

- [ ] Chỉ chấm điểm sân khấu **khác rỗng** (`y^j`)?
- [ ] Sân khấu có nhãn ⇒ nhân `P(x, j)` (không phải `C(x, j)`)?
- [ ] Chặn `j ≤ min(n, x)`?
- [ ] Cập nhật hàng Stirling đúng chiều?
- [ ] Đặt `y = 1` phải ra `x^n`?